# Mission EdStats : réduire le référentiel aux vrais pays

Pour choisir les pays cibles d'academy, il faut d'abord nettoyer le référentiel : `EdStatsCountry.csv` mélange des vrais pays et des agrégats (World, European Union...). Ce notebook construit par le code la liste des vrais pays, `country_codes`. Les résultats du nettoyage iront dans `data/output/`.


## Mise en place + liste des vrais pays

### 1. Dossiers

`INPUT_DIR` pointe vers `data/input/` (lecture seule), `OUTPUT_DIR` vers `data/output/`. Deux chemins candidats selon l'endroit de lancement ; `OUTPUT_DIR` est créé si absent.


In [1]:
import pandas as pd
import pycountry
from pathlib import Path

candidates = [
    Path.cwd() / "data" / "input",         # from the project root
    Path.cwd() / ".." / "data" / "input",  # from notebooks/
]

INPUT_DIR = None
for candidate in candidates:
    if candidate.exists():
        INPUT_DIR = candidate.resolve()
        break
if INPUT_DIR is None:
    raise FileNotFoundError("data/input/ not found")

OUTPUT_DIR = INPUT_DIR.parent / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DIR, OUTPUT_DIR


(PosixPath('/Users/emmanuel/Formation Data Engineer/Projets/P2 - Analyse données systèmes éducatifs/data/input'),
 PosixPath('/Users/emmanuel/Formation Data Engineer/Projets/P2 - Analyse données systèmes éducatifs/data/output'))

### 2. Chargement brut

`load_csv_input()` lit un CSV de `data/input/` tel quel : `utf-8` pour les accents, colonnes « Unnamed » écartées (la virgule finale de chaque ligne crée une colonne vide, c'est un défaut du fichier).


In [2]:
def load_csv_input(name):
    """Read the CSV `name` from INPUT_DIR as is (raw)."""
    return pd.read_csv(
        INPUT_DIR / name,
        encoding="utf-8",
        usecols=lambda column: "Unnamed" not in column,
    )


### 3. Le référentiel

`EdStatsCountry.csv` : une ligne par pays ou agrégat, avec son `Country Code` (ex. `FRA`). Aperçu des 3 premières lignes :


In [3]:
# raw load kept for display only
dfs = {"EdStatsCountry.csv": load_csv_input("EdStatsCountry.csv")}
dfs["EdStatsCountry.csv"].head(3)


,Country Code,Short Name,Table Name,Long Name,2-alpha code,Currency Unit,Special Notes,Region,Income Group,WB-2 code,...,Government Accounting concept,IMF data dissemination standard,Latest population census,Latest household survey,Source of most recent Income and expenditure data,Vital registration complete,Latest agricultural census,Latest industrial data,Latest trade data,Latest water withdrawal data
0,ABW,Aruba,Aruba,Aruba,AW,Aruban florin,SNA data for 2000-2011 are updated from offici...,Latin America & Caribbean,High income: nonOECD,AW,...,NaN,NaN,2010,NaN,NaN,Yes,NaN,NaN,2012.0,NaN
1,AFG,Afghanistan,Afghanistan,Islamic State of Afghanistan,AF,Afghan afghani,Fiscal year end: March 20; reporting period fo...,South Asia,Low income,AF,...,Consolidated central government,General Data Dissemination System (GDDS),1979,"Multiple Indicator Cluster Survey (MICS), 2010/11","Integrated household survey (IHS), 2008",NaN,2013/14,NaN,2012.0,2000
2,AGO,Angola,Angola,People's Republic of Angola,AO,Angolan kwanza,"April 2013 database update: Based on IMF data,...",Sub-Saharan Africa,Upper middle income,AO,...,Budgetary central government,General Data Dissemination System (GDDS),1970,"Malaria Indicator Survey (MIS), 2011","Integrated household survey (IHS), 2008",NaN,2015,NaN,NaN,2005


In [4]:
# separate load for computation
df = load_csv_input("EdStatsCountry.csv")
print(f"{df.shape[0]} rows, {df.shape[1]} columns")


241 rows, 31 columns


### 4. Le critère : un code ISO 3166-1 alpha-3 officiel

`pycountry.countries` est la liste officielle ISO : 249 entrées, chacune avec un code alpha-3 (ex. `FRA`). On en extrait tous les codes dans une liste, puis `isin()` teste chaque ligne : un filtre booléen, aucun `iloc` sur des positions.


In [5]:
# all official ISO 3166-1 alpha-3 codes, as a plain list
iso_codes = []
for country in pycountry.countries:
    iso_codes.append(country.alpha_3)

# boolean filter: True when the row's Country Code is in that list
is_country = df["Country Code"].isin(iso_codes)

country_codes = df.loc[is_country, "Country Code"].tolist()  # the real countries
print(f"{len(df)} rows in the reference, {len(country_codes)} real countries kept")
print(", ".join(sorted(country_codes)))


241 rows in the reference, 214 real countries kept
ABW, AFG, AGO, ALB, AND, ARE, ARG, ARM, ASM, ATG, AUS, AUT, AZE, BDI, BEL, BEN, BFA, BGD, BGR, BHR, BHS, BIH, BLR, BLZ, BMU, BOL, BRA, BRB, BRN, BTN, BWA, CAF, CAN, CHE, CHL, CHN, CIV, CMR, COD, COG, COL, COM, CPV, CRI, CUB, CUW, CYM, CYP, CZE, DEU, DJI, DMA, DNK, DOM, DZA, ECU, EGY, ERI, ESP, EST, ETH, FIN, FJI, FRA, FRO, FSM, GAB, GBR, GEO, GHA, GIB, GIN, GMB, GNB, GNQ, GRC, GRD, GRL, GTM, GUM, GUY, HKG, HND, HRV, HTI, HUN, IDN, IMN, IND, IRL, IRN, IRQ, ISL, ISR, ITA, JAM, JOR, JPN, KAZ, KEN, KGZ, KHM, KIR, KNA, KOR, KWT, LAO, LBN, LBR, LBY, LCA, LIE, LKA, LSO, LTU, LUX, LVA, MAC, MAF, MAR, MCO, MDA, MDG, MDV, MEX, MHL, MKD, MLI, MLT, MMR, MNE, MNG, MNP, MOZ, MRT, MUS, MWI, MYS, NAM, NCL, NER, NGA, NIC, NLD, NOR, NPL, NRU, NZL, OMN, PAK, PAN, PER, PHL, PLW, PNG, POL, PRI, PRK, PRT, PRY, PSE, PYF, QAT, ROU, RUS, RWA, SAU, SDN, SEN, SGP, SLB, SLE, SLV, SMR, SOM, SRB, SSD, STP, SUR, SVK, SVN, SWE, SWZ, SXM, SYC, SYR, TCA, TCD, TGO, THA,

### 5. Les codes écartés

Les lignes écartées, avec leur nom et leur `Region` si renseignée :


In [6]:
excluded = df.loc[~is_country, ["Country Code", "Short Name", "Region"]]
print(f"{len(excluded)} codes excluded, {excluded['Region'].notna().sum()} of them with a Region:")
excluded


27 codes excluded, 2 of them with a Region:


,Country Code,Short Name,Region
5,ARB,Arab World,NaN
35,CHI,Channel Islands,Europe & Central Asia
57,EAP,East Asia & Pacific (developing only),NaN
58,EAS,East Asia & Pacific (all income levels),NaN
59,ECA,Europe & Central Asia (developing only),NaN
60,ECS,Europe & Central Asia (all income levels),NaN
63,EMU,Euro area,NaN
68,EUU,European Union,NaN
89,HIC,High income,NaN
92,HPC,Heavily indebted poor countries (HIPC),NaN


### 6. Deux codes écartés qui ont pourtant une région

`CHI` (Channel Islands) et `XKX` (Kosovo) ont une `Region` mais pas de code ISO officiel. Les deux critères diffèrent sur 4 codes, vus sur les données :


In [7]:
has_region = df["Region"].notna()
print(f"{len(df)} rows: Region set {has_region.sum()}, ISO code {is_country.sum()}, both {(has_region & is_country).sum()}")
print(f"Region yes / ISO no: {df.loc[has_region & ~is_country, 'Country Code'].tolist()}")
print(f"Region no / ISO yes: {df.loc[~has_region & is_country, 'Country Code'].tolist()}")
print(f"neither: {(~has_region & ~is_country).sum()} rows")


241 rows: Region set 214, ISO code 214, both 212
Region yes / ISO no: ['CHI', 'XKX']
Region no / ISO yes: ['GIB', 'NRU']
neither: 25 rows


Cherchons ces noms dans la liste ISO (un dict nom -> code) :


In [8]:
# name -> alpha-3 code for the whole official ISO list
iso_by_name = {}
for c in pycountry.countries:
    iso_by_name[c.name] = c.alpha_3

for name in ["Channel Islands", "Kosovo", "Jersey", "Guernsey"]:
    print(f"{name}: {iso_by_name.get(name, 'not in the official ISO list')}")  # .get(key, default) returns the default if the key is missing


Channel Islands: not in the official ISO list
Kosovo: not in the official ISO list
Jersey: JEY
Guernsey: GGY


Sources externes (liens en section « Sources ») : le Kosovo est un État au statut contesté, « Code ISO 3166-1 : aucun », le code `XK` n'étant « à usage personnel » (Wikipédia, « Kosovo ») ; les îles Anglo-Normandes forment deux États, Jersey et Guernesey (Wikipédia, « Îles Anglo-Normandes »), chacun avec son code ISO officiel (`JEY`, `GGY` ci-dessus). `CHI` est donc un agrégat, `XKX` n'a pas de code officiel : le critère ISO les écarte à raison.


### Bilan

241 lignes en entrée, 214 vrais pays retenus dans `country_codes`, 27 écartés (agrégats + `CHI`/`XKX`, section 6). Prochaine étape : filtrer avec `df["Country Code"].isin(country_codes)` et écrire `data/output/EdStatsCountry.csv`.


## Sources

- Données EdStats (Banque mondiale) : voir `README.md` du projet (`make download`).
- Liste officielle ISO 3166-1 via `pycountry` (249 codes officiels, alpha-2 et alpha-3).
- Wikipédia, « Kosovo » : https://fr.wikipedia.org/wiki/Kosovo
- Wikipédia, « Îles Anglo-Normandes » : https://fr.wikipedia.org/wiki/Îles_Anglo-Normandes
